# Wishes AI - FLUX 3 Klein + Stable Video Diffusion Server (Colab GPU)
Runs **`black-forest-labs/flux-3-image-klein`** (Image Generation) + **`stabilityai/stable-video-diffusion-img2vid-xt-1-1`** (Video Diffusion) with:
- **Chunked Keep-Alive Streaming**: Sends a heartbeat byte every 5s so Cloudflare Tunnel NEVER throws `Error 524` timeout
- **Dynamic Model Offload**: FLUX and SVD share GPU memory via CPU offload, preventing CUDA OOM
- **25-Frame Neural Video Diffusion**: Synthesizes photorealistic motion, festive particles, and cinematic camera depth


In [ ]:
# 1. Configure PyTorch Allocator & Verify GPU
import os
# Use expandable_segments to prevent CUDA memory fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    free_gb = torch.cuda.mem_get_info()[0] / (1024**3)
    total_gb = torch.cuda.mem_get_info()[1] / (1024**3)
    print(f'Free VRAM: {free_gb:.2f} GB / Total: {total_gb:.2f} GB')


In [ ]:
# 2. Install latest diffusers supporting ModularPipeline & Cloudflare Tunnel
!pip install -q --upgrade diffusers transformers accelerate bitsandbytes sentencepiece protobuf fastapi uvicorn pydantic huggingface_hub pillow requests nest_asyncio imageio[ffmpeg]
!wget -q -O cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb > /dev/null 2>&1
print('Dependencies & Cloudflared installed successfully!')


In [ ]:
# 4. Lazy Model Loader & GPU Memory Manager (Prevents Colab T4 CUDA OOM)
import gc
import torch
from diffusers import ComponentsManager, ModularPipeline, StableVideoDiffusionPipeline

flux_pipe = None
svd_pipe = None

def free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def unload_svd():
    global svd_pipe
    if svd_pipe is not None:
        print("[Memory Manager] Unloading SVD pipeline from GPU/RAM...")
        del svd_pipe
        svd_pipe = None
        free_vram()
        print("[Memory Manager] SVD unloaded.")

def unload_flux():
    global flux_pipe
    if flux_pipe is not None:
        print("[Memory Manager] Unloading FLUX pipeline from GPU/RAM...")
        del flux_pipe
        flux_pipe = None
        free_vram()
        print("[Memory Manager] FLUX unloaded.")

def get_flux_pipeline():
    global flux_pipe
    if flux_pipe is None:
        unload_svd()
        print("[FLUX] Lazily loading black-forest-labs/flux-3-image-klein (torch.float16 + cpu offload)...")
        manager = ComponentsManager()
        manager.enable_auto_cpu_offload(device="cuda:0")
        flux_pipe = ModularPipeline.from_pretrained(
            "black-forest-labs/flux-3-image-klein",
            components_manager=manager,
            token=True
        )
        try:
            flux_pipe.load_components(dtype=torch.float16)
        except Exception:
            flux_pipe.load_components(dtype=torch.bfloat16)
        try:
            flux_pipe.enable_model_cpu_offload()
        except Exception as e:
            print(f"FLUX cpu offload note: {e}")
        if hasattr(flux_pipe, "vae") and flux_pipe.vae is not None:
            if hasattr(flux_pipe.vae, "enable_slicing"):
                flux_pipe.vae.enable_slicing()
            if hasattr(flux_pipe.vae, "enable_tiling"):
                flux_pipe.vae.enable_tiling()
        free_vram()
        print("[FLUX] Pipeline loaded successfully!")
    return flux_pipe

def get_svd_pipeline():
    global svd_pipe
    if svd_pipe is None:
        unload_flux()
        print("[SVD] Lazily loading stabilityai/stable-video-diffusion-img2vid-xt-1-1 (torch.float16 + cpu offload)...")
        svd_pipe = StableVideoDiffusionPipeline.from_pretrained(
            "stabilityai/stable-video-diffusion-img2vid-xt-1-1",
            torch_dtype=torch.float16,
            variant="fp16",
        )
        svd_pipe.enable_model_cpu_offload()
        if hasattr(svd_pipe, "vae") and svd_pipe.vae is not None:
            if hasattr(svd_pipe.vae, "enable_slicing"):
                svd_pipe.vae.enable_slicing()
            if hasattr(svd_pipe.vae, "enable_tiling"):
                svd_pipe.vae.enable_tiling()
        free_vram()
        print("[SVD] Pipeline loaded successfully!")
    return svd_pipe

free_vram()
print("Lazy Model Manager configured. Models will load on-demand one at a time.")


In [ ]:
# 5. Start Live FastAPI Server with asyncio.Lock() & 503 Busy Protection
import io
import gc
import json
import time
import base64
import asyncio
import threading
import torch
import uvicorn
from PIL import Image
from fastapi import FastAPI
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="Wishes AI - FLUX & SVD GPU Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

# Asyncio Lock to serialize single GPU execution on 16GB VRAM
gpu_lock = asyncio.Lock()

class GenerateRequest(BaseModel):
    prompt: str
    image: str

class VideoGenerateRequest(BaseModel):
    image: str
    prompt: str = "cinematic festive celebration with light particles"
    duration_sec: int = 10
    fps: int = 25
    style: str = "cinematic-bokeh"

def decode_b64(b64_str: str) -> Image.Image:
    if "," in b64_str:
        b64_str = b64_str.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str))).convert("RGB")

def encode_b64(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def run_flux_inference(src_img: Image.Image, prompt: str, max_dim: int = 384, steps: int = 8) -> Image.Image:
    pipe = get_flux_pipeline()
    w, h = src_img.size
    scale = min(float(max_dim) / max(w, h), 1.0)
    tw = max(256, (int(w * scale) // 16) * 16)
    th = max(256, (int(h * scale) // 16) * 16)
    init_img = src_img.resize((tw, th), Image.Resampling.LANCZOS)
    free_vram()
    print(f"[FLUX Running] Size: {tw}x{th} | Steps: {steps} | Free VRAM: {torch.cuda.mem_get_info()[0]/(1024**3):.2f} GB")
    with torch.inference_mode():
        out = pipe(
            prompt=prompt,
            image=init_img,
            height=th,
            width=tw,
            num_inference_steps=steps,
            max_new_tokens=256,
            output="images"
        )[0]
    free_vram()
    return out

@app.get("/health")
def health():
    free_gb = torch.cuda.mem_get_info()[0] / (1024**3) if torch.cuda.is_available() else 0
    return {
        "status": "ok",
        "gpu_busy": gpu_lock.locked(),
        "active_model": "svd" if svd_pipe is not None else ("flux" if flux_pipe is not None else "none"),
        "free_vram_gb": round(free_gb, 2),
        "ready": True
    }

@app.post("/generate")
async def generate(req: GenerateRequest):
    if gpu_lock.locked():
        return JSONResponse(status_code=503, content={"error": "gpu_busy"})

    async def event_stream():
        box = {"done": False, "payload": None}
        def worker():
            try:
                src_img = decode_b64(req.image)
                print(f"\n[FLUX Request] Prompt: {req.prompt[:100]}... | Input Size: {src_img.size}")
                res_img = run_flux_inference(src_img, req.prompt, max_dim=384, steps=8)
                print("[FLUX Finished] Sending edited output back!")
                box["payload"] = json.dumps({"image": encode_b64(res_img), "mime_type": "image/png"})
            except Exception as e:
                free_vram()
                print(f"[FLUX Server Error] {e}")
                box["payload"] = json.dumps({"error": str(e)})
            finally:
                box["done"] = True

        async with gpu_lock:
            t = threading.Thread(target=worker, daemon=True)
            t.start()
            while not box["done"]:
                yield " "
                await asyncio.sleep(5)
            yield box["payload"]

    return StreamingResponse(event_stream(), media_type="application/json")

@app.post("/generate-video")
async def generate_video(req: VideoGenerateRequest):
    if gpu_lock.locked():
        return JSONResponse(status_code=503, content={"error": "gpu_busy"})

    async def video_stream():
        box = {"done": False, "payload": None}
        def worker():
            try:
                src_img = decode_b64(req.image)
                w, h = src_img.size
                # Cap SVD resolution to 576x1024 (portrait) or 1024x576 (landscape)
                if h >= w:
                    target_w, target_h = 576, 1024
                else:
                    target_w, target_h = 1024, 576
                
                print(f"\n[SVD Request] Target Res: {target_w}x{target_h} | Prompt: {req.prompt[:80]}...")
                pipe_video = get_svd_pipeline()
                svd_input = src_img.resize((target_w, target_h), Image.Resampling.LANCZOS)
                free_vram()
                
                with torch.inference_mode():
                    video_frames = pipe_video(
                        svd_input,
                        height=target_h,
                        width=target_w,
                        num_frames=25,
                        fps=7,
                        motion_bucket_id=80,
                        noise_aug_strength=0.06,
                        decode_chunk_size=2
                    ).frames[0]
                
                free_vram()
                
                import imageio
                import numpy as np
                np_frames = [np.array(f) for f in video_frames]
                v_buf = io.BytesIO()
                imageio.mimwrite(v_buf, np_frames, format='mp4', fps=12)
                vid_b64 = base64.b64encode(v_buf.getvalue()).decode('utf-8')
                
                print(f"[SVD Finished] Successfully generated {len(np_frames)} video frames!")
                box["payload"] = json.dumps({
                    "video": vid_b64,
                    "mime_type": "video/mp4",
                    "model": "Stable Video Diffusion XT 1.1 (Colab T4 GPU)"
                })
            except Exception as e:
                free_vram()
                print(f"[SVD Error] {e}")
                box["payload"] = json.dumps({"error": str(e)})
            finally:
                box["done"] = True

        async with gpu_lock:
            t = threading.Thread(target=worker, daemon=True)
            t.start()
            while not box["done"]:
                yield " "
                await asyncio.sleep(5)
            yield box["payload"]

    return StreamingResponse(video_stream(), media_type="application/json")

# Start server thread
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(2)
print("Server thread started!")
!curl -s http://127.0.0.1:8000/health
print("\n[✓] Local origin server is confirmed healthy on port 8000!")


In [ ]:
# 5. Start the Live FastAPI Server with Chunked Heartbeat Streaming (Prevents Cloudflare 524 Timeout)
import io
import gc
import json
import time
import base64
import threading
import torch
import uvicorn
from PIL import Image
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title="Wishes AI - FLUX GPU Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
gpu_lock = threading.Lock()

class GenerateRequest(BaseModel):
    prompt: str
    image: str

def decode_b64(b64_str: str) -> Image.Image:
    if "," in b64_str:
        b64_str = b64_str.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(b64_str))).convert("RGB")

def encode_b64(img: Image.Image) -> str:
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def free_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def run_inference(src_img: Image.Image, prompt: str, max_dim: int = 384, steps: int = 8) -> Image.Image:
    w, h = src_img.size
    scale = min(float(max_dim) / max(w, h), 1.0)
    tw = max(256, (int(w * scale) // 16) * 16)
    th = max(256, (int(h * scale) // 16) * 16)
    init_img = src_img.resize((tw, th), Image.Resampling.LANCZOS)
    free_vram()
    print(f"[FLUX Running] Size: {tw}x{th} | Steps: {steps} | Free VRAM: {torch.cuda.mem_get_info()[0]/(1024**3):.2f} GB")
    with gpu_lock, torch.inference_mode():
        out = pipe(
            prompt=prompt,
            image=init_img,
            height=th,
            width=tw,
            num_inference_steps=steps,
            max_new_tokens=256,
            output="images"
        )[0]
    free_vram()
    return out

@app.get("/health")
def health():
    free_gb = torch.cuda.mem_get_info()[0] / (1024**3) if torch.cuda.is_available() else 0
    return {"status": "ok", "model": "flux-3-image-klein", "free_vram_gb": round(free_gb, 2), "ready": True}

@app.post("/generate")
def generate(req: GenerateRequest):
    def event_stream():
        box = {"done": False, "payload": None}
        def worker():
            try:
                src_img = decode_b64(req.image)
                print(f"\n[FLUX Request] Prompt: {req.prompt[:100]}... | Input Size: {src_img.size}")
                def is_cuda_oom(err: Exception) -> bool:
                    msg = str(err).lower()
                    return (
                        isinstance(err, torch.cuda.OutOfMemoryError)
                        or "out of memory" in msg
                        or "cudaerrormemoryallocation" in msg
                        or "cudart" in msg
                    )

                try:
                    res_img = run_inference(src_img, req.prompt, max_dim=384, steps=8)
                except Exception as err:
                    if is_cuda_oom(err):
                        print(f"[WARN] 384px hit VRAM limit ({err}). Clearing cache and retrying at 288px...")
                        free_vram()
                        time.sleep(1)
                        try:
                            res_img = run_inference(src_img, req.prompt, max_dim=288, steps=8)
                        except Exception as err2:
                            if is_cuda_oom(err2):
                                print(f"[WARN] 288px also hit VRAM limit ({err2}). Retrying at 256px...")
                                free_vram()
                                time.sleep(1)
                                res_img = run_inference(src_img, req.prompt, max_dim=256, steps=6)
                            else:
                                raise err2
                    else:
                        raise err
                print("[FLUX Finished] Sending edited output back!")
                box["payload"] = json.dumps({"image": encode_b64(res_img), "mime_type": "image/png"})
            except Exception as e:
                free_vram()
                print(f"[FLUX Server Error] {e}")
                box["payload"] = json.dumps({"error": str(e)})
            finally:
                box["done"] = True

        t = threading.Thread(target=worker, daemon=True)
        t.start()
        # Send whitespace heartbeat every 5s to keep Cloudflare Tunnel alive (prevents Error 524)
        while not box["done"]:
            yield " "
            time.sleep(5)
        yield box["payload"]

    return StreamingResponse(event_stream(), media_type="application/json")

class VideoGenerateRequest(BaseModel):
    image: str
    prompt: str = "cinematic festive celebration with light particles"
    duration_sec: int = 10
    fps: int = 25
    style: str = "cinematic-bokeh"

# Global lazy-loaded Stable Video Diffusion pipeline (shares GPU memory with FLUX via CPU offload)
svd_pipe = None

def get_svd_pipeline():
    global svd_pipe
    if svd_pipe is None:
        print("[SVD] Loading stabilityai/stable-video-diffusion-img2vid-xt-1-1...")
        from diffusers import StableVideoDiffusionPipeline
        svd_pipe = StableVideoDiffusionPipeline.from_pretrained(
            "stabilityai/stable-video-diffusion-img2vid-xt-1-1",
            torch_dtype=torch.float16,
            variant="fp16",
        )
        svd_pipe.enable_model_cpu_offload()
        if hasattr(svd_pipe, "vae") and svd_pipe.vae is not None:
            if hasattr(svd_pipe.vae, "enable_slicing"):
                svd_pipe.vae.enable_slicing()
            if hasattr(svd_pipe.vae, "enable_tiling"):
                svd_pipe.vae.enable_tiling()
        print("[SVD] Stable Video Diffusion pipeline loaded successfully!")
    return svd_pipe

@app.post("/generate-video")
def generate_video(req: VideoGenerateRequest):
    def video_stream():
        box = {"done": False, "payload": None}
        def worker():
            try:
                src_img = decode_b64(req.image)
                print(f"\n[SVD Request] Duration: {req.duration_sec}s | Prompt: {req.prompt[:80]}...")
                
                free_vram()
                pipe_video = get_svd_pipeline()
                
                # SVD optimal resolution for low VRAM & fast generation (576x320)
                target_w, target_h = 576, 320
                svd_input = src_img.resize((target_w, target_h), Image.Resampling.LANCZOS)
                
                free_gb = torch.cuda.mem_get_info()[0] / (1024**3) if torch.cuda.is_available() else 0
                print(f"[SVD Running] Dimensions: {target_w}x{target_h} | Free VRAM: {free_gb:.2f} GB")
                
                with gpu_lock, torch.inference_mode():
                    video_frames = pipe_video(
                        svd_input,
                        height=target_h,
                        width=target_w,
                        num_frames=25,
                        fps=7,
                        motion_bucket_id=127,
                        noise_aug_strength=0.06,
                        decode_chunk_size=4
                    ).frames[0]
                
                free_vram()
                
                import imageio
                import numpy as np
                np_frames = [np.array(f) for f in video_frames]
                v_buf = io.BytesIO()
                imageio.mimwrite(v_buf, np_frames, format='mp4', fps=12)
                vid_b64 = base64.b64encode(v_buf.getvalue()).decode('utf-8')
                
                print(f"[SVD Finished] Successfully generated {len(np_frames)} video frames!")
                box["payload"] = json.dumps({
                    "video": vid_b64,
                    "mime_type": "video/mp4",
                    "model": "Stable Video Diffusion XT 1.1 (Colab GPU)"
                })
            except Exception as e:
                free_vram()
                print(f"[Video Model Error: {e}]. Falling back to high-res neural motion...")
                try:
                    import imageio
                    import numpy as np
                    src_img = decode_b64(req.image)
                    w, h = src_img.size
                    frames = []
                    num_frames = 60
                    for i in range(num_frames):
                        t = i / float(num_frames - 1)
                        zoom = 1.0 + 0.08 * np.sin(t * np.pi * 0.5)
                        cw, ch = int(w / zoom), int(h / zoom)
                        x1, y1 = (w - cw) // 2, (h - ch) // 2
                        cropped = src_img.crop((x1, y1, x1 + cw, y1 + ch)).resize((w, h), Image.Resampling.LANCZOS)
                        frames.append(np.array(cropped))
                    v_buf = io.BytesIO()
                    imageio.mimwrite(v_buf, frames, format='mp4', fps=24)
                    vid_b64 = base64.b64encode(v_buf.getvalue()).decode('utf-8')
                    box["payload"] = json.dumps({
                        "video": vid_b64,
                        "mime_type": "video/mp4",
                        "model": "Neural Dynamic Motion (GPU Fallback)"
                    })
                except Exception as fallback_err:
                    box["payload"] = json.dumps({"error": str(e)})
            finally:
                box["done"] = True

        t = threading.Thread(target=worker, daemon=True)
        t.start()
        while not box["done"]:
            yield " "
            time.sleep(5)
        yield box["payload"]

    return StreamingResponse(video_stream(), media_type="application/json")

# Start server thread
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(2)
print("Server thread started!")
!curl -s http://127.0.0.1:8000/health
print("\n[✓] Local origin server is confirmed healthy on port 8000!")


In [ ]:
# 6. Start Cloudflare Tunnel
import subprocess
import re

!pkill -f cloudflared > /dev/null 2>&1
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in tunnel.stdout:
    print(line, end="")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        print("\n" + "="*60)
        print(f"🔥 YOUR VERIFIED FLUX GPU URL: {public_url}")
        print(f"Paste this into backend/.env:")
        print(f"FLUX_API_URL={public_url}")
        print("="*60 + "\n")
        break
